# 🚀 DrumScribe Studio — Servidor GPU Gratuito (Google Colab)

Este notebook transforma o Google Colab no **motor backend com GPU** para o seu site `https://gabriel20gvn.github.io/drumscribe-studio/`.

### Como usar no celular ou PC:
1. No menu superior, confirme **Ambiente de execução → Alterar o tipo de ambiente de execução → T4 GPU**.
2. Clique em **Ambiente de execução → Executar tudo**.
3. Na Célula 2, clique no link azul gerado para abrir o seu site já conectado automaticamente ao servidor!

In [ ]:
#@title 1. Baixar Código do seu GitHub e Instalar Dependências
import os

REPO_URL = "https://github.com/Gabriel20gvn/drumscribe-studio.git"

if not os.path.exists("/content/drumscribe-studio"):
    !git clone {REPO_URL} /content/drumscribe-studio
else:
    %cd /content/drumscribe-studio
    !git pull

%cd /content/drumscribe-studio

!apt-get -qq update && apt-get -qq install -y ffmpeg libsndfile1
!pip install -q -U fastapi "uvicorn[standard]" python-multipart yt-dlp demucs openai-whisper music21
!pip install -q "git+https://github.com/xavriley/ADTOF-pytorch.git@85c192e78f716ea0b111cc8a5ee4a8f6a3a4f8a9"

# Instala o túnel gratuito da Cloudflare (sem necessidade de conta ou token)
if not os.path.exists("/usr/local/bin/cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
    !chmod +x /usr/local/bin/cloudflared

import torch
print("-" * 55)
print("✅ Ambiente pronto!")
print("🚀 GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (ative T4 GPU para máxima velocidade)")

%cd /content
!pip install -q -U "yt-dlp[default]" bgutil-ytdlp-pot-provider
!git clone -q https://github.com/Brainicism/bgutil-ytdlp-pot-provider.git 2>/dev/null || true
%cd /content/bgutil-ytdlp-pot-provider/server
!npm install --silent && npx tsc
!nohup node build/main.js > /tmp/pot_server.log 2>&1 &
%cd /content/drumscribe-studio
!git pull

!curl -fsSL https://deno.land/install.sh | sh -s -- -y >/dev/null 2>&1
import os
os.environ["PATH"] = "/root/.deno/bin:" + os.environ["PATH"]
!pip install -q -U "yt-dlp[default]" pytubefix
%cd /content/drumscribe-studio
!git pull

In [ ]:
#@title 2. Iniciar Servidor Backend e Gerar Link do Site Conectado
#@markdown Mantenha esta célula rodando enquanto estiver usando o site no celular.

import re
import time
import subprocess
from IPython.display import display, HTML

%cd /content/drumscribe-studio

# Encerra instâncias anteriores caso reexecute a célula
!pkill -f "uvicorn app:app" || true
!pkill -f "cloudflared tunnel" || true
time.sleep(1)

# Inicia o FastAPI em segundo plano
api_proc = subprocess.Popen(
    ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

# Inicia o túnel HTTPS público da Cloudflare
cf_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://localhost:7860"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

tunnel_url = None
start_time = time.time()
while time.time() - start_time < 30:
    line = cf_proc.stdout.readline()
    if not line:
        time.sleep(0.2)
        continue
    match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", line)
    if match:
        tunnel_url = match.group(0)
        break

if tunnel_url:
    site_conectado = f"https://gabriel20gvn.github.io/drumscribe-studio/?api={tunnel_url}"
    print("=" * 60)
    print("🟢 SERVIDOR ONLINE!")
    print(f"🔗 Link da API: {tunnel_url}")
    print(f"🌐 Abra seu site conectado: {site_conectado}")
    print("=" * 60)
    display(HTML(f'''
    <div style="padding:16px; background:#0f172a; border:2px solid #22c55e; border-radius:12px; margin-top:10px; font-family:sans-serif;">
      <p style="color:#f8fafc; font-size:16px; margin:0 0 10px 0;">✅ <strong>Backend conectado com sucesso!</strong></p>
      <a href="{site_conectado}" target="_blank" style="display:inline-block; padding:12px 20px; background:#22c55e; color:#052e16; font-weight:bold; text-decoration:none; border-radius:8px; font-size:15px;">
        🥁 Abrir DrumScribe Studio Conectado
      </a>
      <p style="color:#94a3b8; font-size:12px; margin:10px 0 0 0;">Deixe esta aba do Colab aberta em segundo plano enquanto gera as partituras.</p>
    </div>
    '''))
else:
    print("❌ Não foi possível capturar o link do túnel. Tente executar esta célula novamente.")